# Olist Project - Phase 1: Data Cleaning (dataset by dataset)
Keep all 9 raw CSVs in the **same folder** as this notebook and run cells top to bottom.
Each section = one dataset: load, check, clean, save. Cross-table checks come at the very end.

**What:** Import pandas (the only library we need).
**Why:** pandas reads, cleans and saves the CSV files.
**Inconsistency:** None, just setup.

In [1]:
import pandas as pd

## Section 1: Customers

**What:** Load customers and check shape, nulls and duplicates.
**Why:** Always look before cleaning, so we know what is really wrong.
**Inconsistency:** None expected here, this confirms it.

In [5]:
customers = pd.read_csv(
    r'C:\Users\Ankur singh patel\OneDrive\Desktop\Brazillian E-Commerce Project\data_raw\olist_customers_dataset.csv'
)
print(customers.shape)
print('nulls:', customers.isnull().sum().sum())
print('duplicates:', customers.duplicated().sum())

(99441, 5)
nulls: 0
duplicates: 0


**What:** Strip spaces, lowercase city, uppercase state.
**Why:** Otherwise 'Sao Paulo' and 'sao paulo' count as two cities in GROUP BY.
**Inconsistency:** City/state text is not consistent in case.

In [6]:
customers['customer_city'] = customers['customer_city'].str.strip().str.lower()
customers['customer_state'] = customers['customer_state'].str.strip().str.upper()

customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


**What:** Save the clean customers file.
**Why:** You download it and load it into PostgreSQL yourself.
**Inconsistency:** None. `index=False` avoids an extra index column.

In [7]:
customers.to_csv('customers_clean.csv', index=False)
print('customers_clean.csv saved', customers.shape)

customers_clean.csv saved (99441, 5)


## Section 2: Sellers

**What:** Load sellers and check shape, nulls and duplicates.
**Why:** Same first look as every table.
**Inconsistency:** None expected.

In [8]:
sellers = pd.read_csv(
    r'C:\Users\Ankur singh patel\OneDrive\Desktop\Brazillian E-Commerce Project\data_raw\olist_sellers_dataset.csv'
)
print(sellers.shape)
print('nulls:', sellers.isnull().sum().sum())
print('duplicates:', sellers.duplicated().sum())

(3095, 4)
nulls: 0
duplicates: 0


**What:** Same text cleaning as customers (city lower, state upper).
**Why:** We group by seller city/state in SQL and EDA.
**Inconsistency:** Case differences in city names.

In [9]:
sellers['seller_city'] = sellers['seller_city'].str.strip().str.lower()
sellers['seller_state'] = sellers['seller_state'].str.strip().str.upper()

sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


**What:** Save the clean sellers file.
**Why:** Needed for the PostgreSQL load.
**Inconsistency:** None.

In [10]:
sellers.to_csv('sellers_clean.csv', index=False)
print('sellers_clean.csv saved', sellers.shape)

sellers_clean.csv saved (3095, 4)


## Section 3: Geolocation

**What:** Load geolocation and check shape and duplicates.
**Why:** This table is huge, so we must understand it first.
**Inconsistency:** ~1M rows, ~261k exact duplicates.

In [11]:
geolocation = pd.read_csv(
    r'C:\Users\Ankur singh patel\OneDrive\Desktop\Brazillian E-Commerce Project\data_raw\olist_geolocation_dataset.csv'
)
print(geolocation.shape)
print('nulls:', geolocation.isnull().sum().sum())
print('duplicates:', geolocation.duplicated().sum())
print('different zip prefixes:', geolocation['geolocation_zip_code_prefix'].nunique())

(1000163, 5)
nulls: 0
duplicates: 261831
different zip prefixes: 19015


**What:** Make one row per zip prefix (average lat/lng).
**Why:** Many rows per zip would multiply orders when joined (join explosion) and we need one location per zip for distance.
**Inconsistency:** Same zip has many slightly different coordinates.

In [12]:
geolocation['geolocation_city'] = geolocation['geolocation_city'].str.strip().str.lower()

geolocation_clean = geolocation.groupby('geolocation_zip_code_prefix').agg(
    geolocation_lat=('geolocation_lat', 'mean'),
    geolocation_lng=('geolocation_lng', 'mean'),
    geolocation_city=('geolocation_city', 'first'),
    geolocation_state=('geolocation_state', 'first')
).reset_index()

print('Before:', geolocation.shape, ' After:', geolocation_clean.shape)

Before: (1000163, 5)  After: (19015, 5)


**What:** Remove zips whose coordinates fall outside Brazil.
**Why:** A wrong coordinate gives a wrong seller-to-customer distance later.
**Inconsistency:** 11 zip prefixes have coordinates outside Brazil.

In [13]:
print('Before:', geolocation_clean.shape)

geolocation_clean = geolocation_clean[(geolocation_clean['geolocation_lat'] <= 5.5) & (geolocation_clean['geolocation_lat'] >= -34)]
geolocation_clean = geolocation_clean[(geolocation_clean['geolocation_lng'] <= -34) & (geolocation_clean['geolocation_lng'] >= -74)]

print('After:', geolocation_clean.shape)

Before: (19015, 5)
After: (19004, 5)


**What:** Save the clean geolocation file.
**Why:** Needed for the PostgreSQL load and for the distance feature in ML.
**Inconsistency:** None.

In [14]:
geolocation_clean.to_csv('geolocation_clean.csv', index=False)
print('geolocation_clean.csv saved', geolocation_clean.shape)

geolocation_clean.csv saved (19004, 5)


## Section 4: Products (with English category names)

**What:** Load products and the translation file, check nulls.
**Why:** Category names are Portuguese; the translation file gives English names.
**Inconsistency:** 610 products miss category/info, 2 miss weight and size.

In [15]:
products = pd.read_csv(
    r'C:\Users\Ankur singh patel\OneDrive\Desktop\Brazillian E-Commerce Project\data_raw\olist_products_dataset.csv'
)
translation = pd.read_csv(
    r'C:\Users\Ankur singh patel\OneDrive\Desktop\Brazillian E-Commerce Project\data_raw\product_category_name_translation.csv'
)
print(products.shape, translation.shape)
print('duplicates:', products.duplicated().sum())
products.isnull().sum()

(32951, 9) (71, 2)
duplicates: 0


product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

**What:** Fix misspelled column names and fill missing category with 'unknown'.
**Why:** `lenght` is a typo; null categories would break group-by and we do not want to lose those products.
**Inconsistency:** Typo in 2 column names, 610 missing categories.

In [16]:
products = products.rename(columns={
    'product_name_lenght': 'product_name_length',
    'product_description_lenght': 'product_description_length'
})

products['product_category_name'] = products['product_category_name'].fillna('unknown')

products.columns

Index(['product_id', 'product_category_name', 'product_name_length',
       'product_description_length', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='object')

**What:** Fill missing numeric columns with the median.
**Why:** Median is simple and not pulled by very heavy or large products.
**Inconsistency:** 610 rows miss length/photos info, 2 rows miss weight and size.

In [17]:
products['product_name_length'] = products['product_name_length'].fillna(products['product_name_length'].median())
products['product_description_length'] = products['product_description_length'].fillna(products['product_description_length'].median())
products['product_photos_qty'] = products['product_photos_qty'].fillna(products['product_photos_qty'].median())
products['product_weight_g'] = products['product_weight_g'].fillna(products['product_weight_g'].median())
products['product_length_cm'] = products['product_length_cm'].fillna(products['product_length_cm'].median())
products['product_height_cm'] = products['product_height_cm'].fillna(products['product_height_cm'].median())
products['product_width_cm'] = products['product_width_cm'].fillna(products['product_width_cm'].median())

print('nulls left:', products.isnull().sum().sum())

nulls left: 0


**What:** Replace product weight 0 with the median weight.
**Why:** No real product weighs 0 g and weight is an important ML feature.
**Inconsistency:** 4 products have weight 0, which is a hidden missing value (isnull cannot see it).

In [18]:
print('Products with weight 0:', (products['product_weight_g'] == 0).sum())

products.loc[products['product_weight_g'] == 0, 'product_weight_g'] = products['product_weight_g'].median()

print('Products with weight 0 now:', (products['product_weight_g'] == 0).sum())

Products with weight 0: 4
Products with weight 0 now: 0


**What:** Add the English category name using a left join.
**Why:** English names are needed for charts and the dashboard; left join keeps every product.
**Inconsistency:** 2 categories are missing in the translation file, so we use the Portuguese name for them.

In [19]:
products_clean = products.merge(translation, on='product_category_name', how='left')

products_clean['product_category_name_english'] = products_clean['product_category_name_english'].fillna(products_clean['product_category_name'])

print(products_clean.shape, ' nulls:', products_clean.isnull().sum().sum())

(32951, 10)  nulls: 0


**What:** Save the clean products file.
**Why:** Needed for the PostgreSQL load.
**Inconsistency:** None. 2 raw files became 1 clean file.

In [20]:
products_clean.to_csv('products_clean.csv', index=False)
print('products_clean.csv saved', products_clean.shape)

products_clean.csv saved (32951, 10)


## Section 5: Orders

**What:** Load orders and check nulls and duplicates.
**Why:** Orders is the main table of the project.
**Inconsistency:** Missing approval, carrier and delivery dates.

In [21]:
orders = pd.read_csv(
    r'C:\Users\Ankur singh patel\OneDrive\Desktop\Brazillian E-Commerce Project\data_raw\olist_orders_dataset.csv'
)
print(orders.shape)
print('duplicates:', orders.duplicated().sum())
orders.isnull().sum()

(99441, 8)
duplicates: 0


order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

**What:** Convert the 5 date columns from text to datetime.
**Why:** Late delivery needs date comparison, which text cannot do.
**Inconsistency:** Dates are plain strings in the raw file.

In [22]:
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_approved_at'] = pd.to_datetime(orders['order_approved_at'])
orders['order_delivered_carrier_date'] = pd.to_datetime(orders['order_delivered_carrier_date'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])

orders.dtypes

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

**What:** Check which order statuses have a missing delivery date.
**Why:** To decide whether the nulls are errors or genuine.
**Inconsistency:** ~2,965 missing dates, but almost all belong to canceled/shipped orders (only 8 are 'delivered'), so we do NOT fill them. The 8 are dropped in the ML phase.

In [23]:
print(orders['order_status'].value_counts())
print()
print(orders[orders['order_delivered_customer_date'].isnull()]['order_status'].value_counts())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


**What:** Save the clean orders file.
**Why:** Needed for the PostgreSQL load.
**Inconsistency:** Nulls in delivery dates stay (genuinely not delivered).

In [24]:
orders.to_csv('orders_clean.csv', index=False)
print('orders_clean.csv saved', orders.shape)

orders_clean.csv saved (99441, 8)


## Section 6: Order Items

**What:** Load order items and check nulls and duplicates.
**Why:** This table holds price and freight, which are key ML features.
**Inconsistency:** None expected.

In [25]:
items = pd.read_csv(
    r'C:\Users\Ankur singh patel\OneDrive\Desktop\Brazillian E-Commerce Project\data_raw\olist_order_items_dataset.csv'
)
print(items.shape)
print('nulls:', items.isnull().sum().sum())
print('duplicates:', items.duplicated().sum())

(112650, 7)
nulls: 0
duplicates: 0


**What:** Convert shipping_limit_date to datetime.
**Why:** It is a date but stored as text.
**Inconsistency:** Wrong data type only. Price and freight outliers are real, so we keep them.

In [26]:
items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'])

items.dtypes

order_id                       object
order_item_id                   int64
product_id                     object
seller_id                      object
shipping_limit_date    datetime64[ns]
price                         float64
freight_value                 float64
dtype: object

**What:** Save the clean order items file.
**Why:** Needed for the PostgreSQL load.
**Inconsistency:** None.

In [27]:
items.to_csv('order_items_clean.csv', index=False)
print('order_items_clean.csv saved', items.shape)

order_items_clean.csv saved (112650, 7)


## Section 7: Order Payments

**What:** Load payments and look at the payment types.
**Why:** To spot invalid payment types.
**Inconsistency:** 3 rows have type 'not_defined' with zero value.

In [28]:
payments = pd.read_csv(
    r'C:\Users\Ankur singh patel\OneDrive\Desktop\Brazillian E-Commerce Project\data_raw\olist_order_payments_dataset.csv'
)
print(payments.shape)
print('nulls:', payments.isnull().sum().sum(), ' duplicates:', payments.duplicated().sum())
print(payments['payment_type'].value_counts())

(103886, 5)
nulls: 0  duplicates: 0
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


**What:** Check the 'not_defined' rows, but keep them.
**Why:** They are the only payment row of 3 orders, so deleting them would leave those orders without a payment record. Their value is 0, so totals do not change.
**Inconsistency:** Undefined payment type with zero amount (filter it in SQL/EDA if needed).

In [29]:
payments_clean = payments.copy()

print(payments_clean[payments_clean['payment_type'] == 'not_defined'])
print('Rows kept:', payments_clean.shape)

                               order_id  payment_sequential payment_type  \
51280  4637ca194b6387e2d538dc89b124b0ee                   1  not_defined   
57411  00b1cb0320190ca0daa2c88b35206009                   1  not_defined   
94427  c8c528189310eaa44a745b8d9d26908b                   1  not_defined   

       payment_installments  payment_value  
51280                     1            0.0  
57411                     1            0.0  
94427                     1            0.0  
Rows kept: (103886, 5)


**What:** Save the clean payments file.
**Why:** Needed for the PostgreSQL load.
**Inconsistency:** None. One order can have several payment rows, we handle that in SQL.

In [30]:
payments_clean.to_csv('order_payments_clean.csv', index=False)
print('order_payments_clean.csv saved', payments_clean.shape)

order_payments_clean.csv saved (103886, 5)


## Section 8: Order Reviews

**What:** Load reviews and check nulls and duplicates.
**Why:** To see how empty the comment columns are.
**Inconsistency:** 87,656 empty titles and 58,247 empty messages.

In [32]:
reviews = pd.read_csv(
    r'C:\Users\Ankur singh patel\OneDrive\Desktop\Brazillian E-Commerce Project\data_raw\olist_order_reviews_dataset.csv'
)
print(reviews.shape)
print('duplicates:', reviews.duplicated().sum())
reviews.isnull().sum()

(99224, 7)
duplicates: 0


review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

**What:** Drop the comment columns and convert review dates.
**Why:** Comments are mostly empty and we do no text analysis; only the score is needed.
**Inconsistency:** Heavy nulls in comments, dates stored as text.

In [33]:
reviews_clean = reviews.drop(columns=['review_comment_title', 'review_comment_message'])

reviews_clean['review_creation_date'] = pd.to_datetime(reviews_clean['review_creation_date'])
reviews_clean['review_answer_timestamp'] = pd.to_datetime(reviews_clean['review_answer_timestamp'])

reviews_clean.dtypes

review_id                          object
order_id                           object
review_score                        int64
review_creation_date       datetime64[ns]
review_answer_timestamp    datetime64[ns]
dtype: object

**What:** Keep only the latest review per order.
**Why:** An order with 2 reviews would appear twice after a join and double its revenue.
**Inconsistency:** The same order_id appears more than once (551 extra reviews).

In [34]:
print('Orders with more than one review:', reviews_clean['order_id'].duplicated().sum())

reviews_clean = reviews_clean.sort_values('review_answer_timestamp')
reviews_clean = reviews_clean.drop_duplicates(subset='order_id', keep='last')

print('After:', reviews_clean.shape)

Orders with more than one review: 551
After: (98673, 5)


**What:** Save the clean reviews file.
**Why:** Needed for the PostgreSQL load.
**Inconsistency:** None.

In [35]:
reviews_clean.to_csv('order_reviews_clean.csv', index=False)
print('order_reviews_clean.csv saved', reviews_clean.shape)

order_reviews_clean.csv saved (98673, 5)


## Section 9: Cross-table checks (after all datasets are done)
These checks need more than one table, so they come last. We only **check and count** here. Orders with problems are real purchases, so we keep them for SQL/EDA and exclude them only in the ML phase.

**What:** Check impossible date order in orders (delivered orders only for the last check).
**Why:** A pickup before the purchase, or delivery before pickup, cannot happen in real life.
**Inconsistency:** ~166 orders with pickup before purchase and ~23 delivered before pickup.

In [36]:
delivered = orders[orders['order_status'] == 'delivered']

print('pickup before purchase:', (orders['order_delivered_carrier_date'] < orders['order_purchase_timestamp']).sum())
print('delivered before pickup:', (delivered['order_delivered_customer_date'] < delivered['order_delivered_carrier_date']).sum())
print('delivered but no delivery date:', delivered['order_delivered_customer_date'].isnull().sum())

pickup before purchase: 166
delivered before pickup: 23
delivered but no delivery date: 8


**What:** Check that customer and seller zips exist in the clean geolocation table.
**Why:** Without a match the distance feature will be empty for that order.
**Inconsistency:** ~287 customers and ~7 sellers have a zip not found in geolocation.

In [37]:
print('customer zips not in geolocation:', (~customers['customer_zip_code_prefix'].isin(geolocation_clean['geolocation_zip_code_prefix'])).sum())
print('seller zips not in geolocation:', (~sellers['seller_zip_code_prefix'].isin(geolocation_clean['geolocation_zip_code_prefix'])).sum())

customer zips not in geolocation: 287
seller zips not in geolocation: 7


**What:** Check that keys match across tables (orders vs customers, items, payments, reviews).
**Why:** Joins in SQL only work if the ids exist on both sides.
**Inconsistency:** Some orders have no items (~775, mostly canceled), which is expected.

In [38]:
print('orders without customer:', (~orders['customer_id'].isin(customers['customer_id'])).sum())
print('orders without items:', (~orders['order_id'].isin(items['order_id'])).sum())
print('orders without payment:', (~orders['order_id'].isin(payments_clean['order_id'])).sum())
print('items with unknown product:', (~items['product_id'].isin(products_clean['product_id'])).sum())
print('items with unknown seller:', (~items['seller_id'].isin(sellers['seller_id'])).sum())

orders without customer: 0
orders without items: 775
orders without payment: 1
items with unknown product: 0
items with unknown seller: 0


**What:** Final summary of all 8 clean tables (rows and total nulls).
**Why:** Confirms cleaning is complete. Only date nulls in orders should remain.
**Inconsistency:** Remaining nulls in orders are expected (undelivered orders).

In [39]:
print('customers:', customers.shape, customers.isnull().sum().sum())
print('sellers:', sellers.shape, sellers.isnull().sum().sum())
print('geolocation:', geolocation_clean.shape, geolocation_clean.isnull().sum().sum())
print('products:', products_clean.shape, products_clean.isnull().sum().sum())
print('orders:', orders.shape, orders.isnull().sum().sum())
print('items:', items.shape, items.isnull().sum().sum())
print('payments:', payments_clean.shape, payments_clean.isnull().sum().sum())
print('reviews:', reviews_clean.shape, reviews_clean.isnull().sum().sum())

customers: (99441, 5) 0
sellers: (3095, 4) 0
geolocation: (19004, 5) 0
products: (32951, 10) 0
orders: (99441, 8) 4908
items: (112650, 7) 0
payments: (103886, 5) 0
reviews: (98673, 5) 0
